In [1]:
import numpy as np
import pandas as pd
import gudhi as gd
import matplotlib.pyplot as plt
import networkx as nx
import ipywidgets as widgets
from IPython.display import display

In [2]:
def load_distance_matrix(file_name):
    return pd.read_excel(file_name, index_col=0).values

def compute_rips_from_distance_matrix(distance_matrix, max_edge_length):
    rips_complex = gd.RipsComplex(distance_matrix=distance_matrix, max_edge_length=max_edge_length)
    simplex_tree = rips_complex.create_simplex_tree(max_dimension=2)
    return simplex_tree

def extract_simplices(simplex_tree):
    simplices = {'vertices': set(), 'edges': [], 'faces': []}
    for simplex, _ in simplex_tree.get_filtration():
        if len(simplex) == 1:
            simplices['vertices'].add(simplex[0])
        elif len(simplex) == 2:
            simplices['edges'].append(tuple(simplex))
        elif len(simplex) == 3:
            simplices['faces'].append(tuple(simplex))
    return simplices

def compute_persistence(simplex_tree):
    parent = {}
    birth_time = {}
    birth_death_dict = {}
    
    for simplex, filtration in simplex_tree.get_filtration():
        if len(simplex) == 1:
            v = simplex[0]
            parent[v] = v
            birth_time[v] = 0
            birth_death_dict[v] = (0, None)
    
    def find(v):
        if parent[v] != v:
            parent[v] = find(parent[v])
        return parent[v]
    
    def union(v1, v2, death_time):
        root1 = find(v1)
        root2 = find(v2)
        if root1 != root2:
            if birth_time[root1] < birth_time[root2]:  
                parent[root2] = root1
                birth_death_dict[root2] = (birth_death_dict[root2][0], death_time)
            else:
                parent[root1] = root2
                birth_death_dict[root1] = (birth_death_dict[root1][0], death_time)
    
    for simplex, filtration in simplex_tree.get_filtration():
        if len(simplex) == 2:
            v1, v2 = simplex
            union(v1, v2, filtration)
    
    return birth_death_dict

def store_results_in_excel(distance_matrix, file_name):
    max_edge_length = np.max(distance_matrix)
    simplex_tree = compute_rips_from_distance_matrix(distance_matrix, max_edge_length)
    simplices = extract_simplices(simplex_tree)
    birth_death_dict = compute_persistence(simplex_tree)
    
    #vertices_df = pd.DataFrame({'Vertices': list(simplices['vertices'])})
    #edges_df = pd.DataFrame(simplices['edges'], columns=['Vertex 1', 'Vertex 2'])
    #faces_df = pd.DataFrame(simplices['faces'], columns=['Vertex 1', 'Vertex 2', 'Vertex 3'])
    persistence_df = pd.DataFrame([(k, v[0], v[1] if v[1] else 'infinity') for k, v in birth_death_dict.items()],
                                  columns=['Protein', 'Birth', 'Death'])
    
    with pd.ExcelWriter(file_name) as writer:
        #vertices_df.to_excel(writer, sheet_name='Vertices', index=False)
        #edges_df.to_excel(writer, sheet_name='Edges', index=False)
        #faces_df.to_excel(writer, sheet_name='Faces', index=False)
        persistence_df.to_excel(writer, sheet_name='Persistence', index=False)

def main():
    distance_matrix_file = "trial_distance_matrix_800v1.xlsx"
    distance_matrix = load_distance_matrix(distance_matrix_file)
    output_file = "h0_results_800v1.xlsx"
    store_results_in_excel(distance_matrix, output_file)
    print(f"Results stored in {output_file}")

if __name__ == "__main__":
    main()


Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "C:\Users\Erich Faye Rinon\AppData\Roaming\Python\Python311\site-packages\IPython\core\interactiveshell.py", line 3526, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "C:\Users\Erich Faye Rinon\AppData\Local\Temp\ipykernel_23784\3304147990.py", line 81, in <module>
    main()
  File "C:\Users\Erich Faye Rinon\AppData\Local\Temp\ipykernel_23784\3304147990.py", line 77, in main
    store_results_in_excel(distance_matrix, output_file)
  File "C:\Users\Erich Faye Rinon\AppData\Local\Temp\ipykernel_23784\3304147990.py", line 58, in store_results_in_excel
    simplices = extract_simplices(simplex_tree)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Erich Faye Rinon\AppData\Local\Temp\ipykernel_23784\3304147990.py", line 17, in extract_simplices
    simplices['faces'].append(tuple(simplex))
                              ^^^^^^^^^^^^^^
MemoryError

During handling of the above exception, another exception occ

In [4]:
distance_matrix_file = "trial_distance_matrix_500.xlsx"
distance_matrix = load_distance_matrix(distance_matrix_file)
np.max(distance_matrix)

1.414213562373095